## Processing large files from pubtator3
*OpenAI used as basis*

## To-do:
switch sqlite tables to temporary tables 
prelim processing from temporary tables
only transfer to regular tables if needed else delete
how to combine info stored as json and sqlite?
how to get only top info from sqlite?

### Package organization: 
https://www.ncbi.nlm.nih.gov/CBBresearch/Lu/Demo/tmTools/Format.html

## Package installation

In [38]:
import sys
print(sys.executable)
!{sys.executable} -m pip install lxml
!{sys.executable} -m pip install tqdm
!{sys.executable} -m pip install scipy

/modules/opt/linux-ubuntu24.04-x86_64/jupyterlab/unity-jupyterlab4.4.3/bin/python
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 96.2 MB/s eta 0:00:00ta 0:00:01


In [23]:
#!which pip
#!pip --version

In [24]:
#!pip install lxml

In [12]:
import requests
import tarfile
import gzip 
import io
import json
from lxml import etree #both of these should be fine but aren't working 
from tqdm.auto import tqdm #"no module named"
import time
import sqlite3
from typing import Optional
import os
import sys
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import scipy

## Configuration 

In [13]:
URL = "https://ftp.ncbi.nlm.nih.gov/pub/lu/PubTator3/BioCXML.0.tar.gz"
URLS = ["https://ftp.ncbi.nlm.nih.gov/pub/lu/PubTator3/BioCXML.0.tar.gz", "https://ftp.ncbi.nlm.nih.gov/pub/lu/PubTator3/BioCXML.1.tar.gz", 
        "https://ftp.ncbi.nlm.nih.gov/pub/lu/PubTator3/BioCXML.2.tar.gz", "https://ftp.ncbi.nlm.nih.gov/pub/lu/PubTator3/BioCXML.3.tar.gz", 
        "https://ftp.ncbi.nlm.nih.gov/pub/lu/PubTator3/BioCXML.4.tar.gz", "https://ftp.ncbi.nlm.nih.gov/pub/lu/PubTator3/BioCXML.5.tar.gz", 
        "https://ftp.ncbi.nlm.nih.gov/pub/lu/PubTator3/BioCXML.6.tar.gz", "https://ftp.ncbi.nlm.nih.gov/pub/lu/PubTator3/BioCXML.7.tar.gz",
        "https://ftp.ncbi.nlm.nih.gov/pub/lu/PubTator3/BioCXML.8.tar.gz", "https://ftp.ncbi.nlm.nih.gov/pub/lu/PubTator3/BioCXML.9.tar.gz"]
SQLITE_PATH = "pubtator_bioc0.sqlite" # will I need more for later? or is it useful to have different tables on different documents? 
MEMBER_LIMIT = 10           # set to None to process all members
DOCS_COMMIT_BATCH = 200     # commit every N documents
PROCESS_ONLY_XML = True     # if True, skip non-XML members
SHOW_PROGRESS = True

## SQLite schema and initialization

In [5]:
CREATE_TABLES_SQL = """
PRAGMA journal_mode = WAL;
PRAGMA synchronous = NORMAL;

CREATE TABLE IF NOT EXISTS documents (
    doc_id TEXT PRIMARY KEY,
    member_name TEXT,        -- which archive member the document came from
    infons TEXT,             -- JSON
    title TEXT, 
    abstract TEXT,
    author TEXT,
    doc_type TEXT,
    contains_a INT,
    contains_c INT
);

CREATE TABLE IF NOT EXISTS a_documents (
    doc_id TEXT PRIMARY KEY,
    member_name TEXT,        -- which archive member the document came from
    infons TEXT,             -- JSON
    title TEXT, 
    abstract TEXT,
    author TEXT,
    doc_type TEXT,
    contains_a INT,
    contains_c INT
);

CREATE TABLE IF NOT EXISTS b_documents (
    doc_id TEXT PRIMARY KEY,
    member_name TEXT,        -- which archive member the document came from
    infons TEXT,             -- JSON
    title TEXT, 
    abstract TEXT,
    author TEXT,
    doc_type TEXT,
    contains_a INT,
    contains_c INT
);

CREATE TABLE IF NOT EXISTS c_documents (
    doc_id TEXT PRIMARY KEY,
    member_name TEXT,        -- which archive member the document came from
    infons TEXT,             -- JSON
    title TEXT, 
    abstract TEXT,
    author TEXT,
    doc_type TEXT,
    contains_a INT,
    contains_c INT
);

CREATE TABLE IF NOT EXISTS overlap_documents (
    doc_id TEXT PRIMARY KEY,
    member_name TEXT,        -- which archive member the document came from
    infons TEXT,             -- JSON
    title TEXT, 
    abstract TEXT,
    author TEXT,
    doc_type TEXT,
    contains_a INT,
    contains_c INT
);

CREATE TABLE IF NOT EXISTS a_terms (
    searchA TEXT PRIMARY KEY,
    synonyms TEXT        -- as list 
);

CREATE TABLE IF NOT EXISTS b_terms (
    searchA TEXT PRIMARY KEY,
    synonyms TEXT,
    doc_id TEXT        -- will likely add more to this later but this is good to start
);

CREATE TABLE IF NOT EXISTS c_terms (
    searchA TEXT PRIMARY KEY,
    synonyms TEXT 
);

CREATE TABLE IF NOT EXISTS passages (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    doc_id TEXT,
    member_name TEXT,
    offset INTEGER,
    infons TEXT,
    text TEXT
);

CREATE TABLE IF NOT EXISTS sentences (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    passage_id INTEGER,
    infons TEXT,
    text TEXT
);

CREATE TABLE IF NOT EXISTS annotations (
    ann_id TEXT PRIMARY KEY,
    doc_id TEXT,
    passage_id INTEGER,
    sentence_id TEXT,
    location_offset INTEGER,
    location_length INTEGER,
    type TEXT,
    text TEXT,
    infons TEXT
);

CREATE TABLE IF NOT EXISTS relations (
    rel_id TEXT PRIMARY KEY,
    doc_id TEXT,
    passage_id INTEGER,
    sentence_id INTEGER,
    infons TEXT
);

CREATE TABLE IF NOT EXISTS relation_nodes (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    rel_id TEXT,
    refid TEXT,
    role TEXT
);

-- bookkeeping: record processed member names [have seen before, not necessarily designated as relevant]so we can resume
-- would be useful to have further bookkeeping for separate searches 
CREATE TABLE IF NOT EXISTS processed_members (
    member_name TEXT PRIMARY KEY,
    processed_at REAL
);

CREATE INDEX IF NOT EXISTS idx_passages_doc ON passages(doc_id);
CREATE INDEX IF NOT EXISTS idx_annotations_doc ON annotations(doc_id);
"""


In [15]:
def init_db(path: str):
    conn = sqlite3.connect(path, timeout=60)
    cur = conn.cursor()
    cur.executescript(CREATE_TABLES_SQL)
    conn.commit()
    return conn

In [6]:
# Small XML helper utilities
## finding a small XML file to test on may be helpful for these as there seem to be some glitches 
def localname(tag: Optional[str]) -> Optional[str]:
    if tag is None:
        return None
    return tag.split("}")[-1] if "}" in tag else tag # split at location right before }? 

def get_child_text(elem, child_name: str) -> Optional[str]:
    for ch in elem:
        if child_name in localname(ch.tag):
            return (ch.text or ch.get or "").strip()
    return None

def get_children(elem, child_name: str):
    for ch in elem:
        if localname(ch.tag) == child_name:
            yield ch

def infons_to_dict(parent_elem):
    d = {}
    for inf in get_children(parent_elem, "infon"):
        if "key" in inf.attrib:
            key = inf.attrib["key"]
            d[key] = (inf.text or "").strip()
        else:
            # fallback: accumulate unnamed infons
            v = (inf.text or "").strip()
            if v:
                d.setdefault("notes", []).append(v)
    return d

## parse_member_and_insert

In [7]:
# Core: parse one archive member (BioC XML) incrementally and insert into DB
def parse_member_and_insert(member_fileobj, member_name, termA, termC: str, conn: sqlite3.Connection,
                            docs_commit_batch: int = DOCS_COMMIT_BATCH,
                            max_docs: Optional[int]=None,
                            show_progress: bool = SHOW_PROGRESS):
    """
    member_fileobj: binary file-like for the member bytes
    member_name: name string (used for bookkeeping)
    conn: sqlite connection
    max_docs: optional limit of documents to process for this member (useful for testing)
    Returns: number of documents processed
    """
    # wrap gzip if member itself is gz inside the tar
    if member_name.endswith(".gz"):
        binstream = gzip.GzipFile(fileobj=member_fileobj)
    else:
        binstream = member_fileobj

    parser = etree.XMLParser(recover=True, huge_tree=True)
    context = etree.iterparse(binstream, events=("end",))

    cur = conn.cursor()
    docs = 0
    t0 = time.time()

    try:
        for event, elem in context:
            #print("localname(elem.tag):", localname(elem.tag))
            #if localname(elem.tag) and localname(elem.tag).lower() == "document": # doesn't seem to make a difference
            if localname(elem.tag) == "document":
                # children are id and passages
                # document id
                #print(get_child_text(elem, "id"))
                doc_id = get_child_text(elem, "id") or get_child_text(elem, "pmid") or None
                doc_infons = infons_to_dict(elem)
                #print(doc_infons)
                title_text = None
                abstract_text = None
                doc_type = get_child_text(elem, "type") or None
                author = get_child_text(elem, "author") or None #get same way as title but may need to iterate and parse first vs last
                contains_a = 0
                contains_c = 0

                # insert document row
                cur.execute( 
                    "INSERT OR REPLACE INTO documents(doc_id, member_name, infons, title, abstract, doc_type, author, contains_a, contains_c) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)",
                    (doc_id, member_name, json.dumps(doc_infons, ensure_ascii=False), title_text, None, None, author, contains_a, contains_c)
                )

                # process passages
                for passage in (ch for ch in elem if localname(ch.tag) == "passage"):
                    #passage is not a format that can be printed
                    p_infons = infons_to_dict(passage)
                    # print(p_infons) # this truly does have ALL of the info
                    # offset
                    offset = None
                    off = get_child_text(passage, "offset")
                    if off and off.isdigit():
                        offset = int(off)
                    else:
                        for loc in get_children(passage, "location"):
                            if "offset" in loc.attrib:
                                try:
                                    offset = int(loc.attrib.get("offset"))
                                except Exception:
                                    pass
                                break
                    p_text = get_child_text(passage, "text") or ""

                    cur.execute(
                        "INSERT INTO passages(doc_id, member_name, offset, infons, text) VALUES (?, ?, ?, ?, ?)",
                        (doc_id, member_name, offset, json.dumps(p_infons, ensure_ascii=False), p_text)
                    )
                    passage_rowid = cur.lastrowid 

                    # heuristics for title/abstract
                    # section_type is TITLE, ABSTRACT, INTRO, METHODS
                    # type is title_1, title_2, paragraph
                    pstype = p_infons.get("section_type") 
                    ptype = p_infons.get("type")
                    if pstype:
                        if (pstype.lower() in "title" or pstype.lower() in "article_title"):
                            title_text = p_text
                            if termA in title_text.lower():
                                contains_a += 1
                            if termC in title_text.lower():
                                contains_c += 1
                        elif (pstype.lower() in "abstract" or pstype.lower() in "abstract_text"):
                            abstract_text = (abstract_text + "\n" + p_text) if abstract_text else p_text
                        elif ptype.lower() in ("paragraph passage sentence text"):
                            paragraph_text = p_text
                        elif ptype.lower() in ("annotation"):
                            annotation_text = p_text
                        elif ptype.lower() in ("relation"):
                            relation_text = p_text

                    #if termA in ptype.lower():
                        #print("Search Term A found in ptype")
                        # want to update documents to show it contains the term
                        # doubt that it would really show up in this category but I'm trying all of the things
                    #if termC in ptype.lower():
                        #print("Search Term C found in ptype")
                        
                    if termA in p_text.lower():
                        #print("Search Term A found in p_text")
                        contains_a += 1
                        # want to update documents to show it contains the term
                        # doubt that it would really show up in this category but I'm trying all of the things
                    if termC in p_text.lower():
                        #print("Search Term C found in p_text")
                        contains_c += 1
                    # is running code here
                    #print(passage.tag) >> passage
                    #print(passage) >> nothing ??
                    
                    # sentences
                    # i = 0
                    for sentence in (s for s in passage if ptype.lower() in ("paragraph annotation relation")):
                        # no type labeled sentence, but are types labeled paragraph. "text" doesn't work 
                        s_infons = infons_to_dict(sentence)
                        #print(s_infons) # gets annotations but not text
                        s_text = p_text.lower().split('.')
                        # while i < sentence.length():
                        #     s_text = p_text.lower().split('.')[i]
                        #     i+=1
                        # why can't i just use sssssss 
                        #s_text = get_child_text(sentence, "text") or get_child_text(sentence, "paragraph") or get_child_text(sentence, "sentence") or ""
                        #print("Found sentence: " + sentence)
                        cur.execute(
                            "INSERT INTO sentences(passage_id, infons, text) VALUES (?, ?, ?)",
                            (passage_rowid, None, s_text[0])
                        ) 
                        # somehow [0] gets a variety of rows?? confusion but if it works it works
                        # okay stopped working now annotations are somewhat working
                        sentence_rowid = cur.lastrowid
                        #print(sentence_rowid) #we know this is working through but it's not getting into sentences
                        stype = s_infons.get("type") or s_infons.get("section")

                        # annotations inside sentence
                        for ann in (a for a in sentence if stype.lower() in ("chemical species disease gene")):
                            ann_text = get_child_text(sentence, "text") or get_child_text(ann, "text") or None
                            ann_infons = s_infons
                            ann_id = get_child_text(sentence, "id") or get_child_text(ann, "id") or get_child_text(ann, "identifier") or None
                            #ann_id = ann_infons.get("identifier") or ann_infons.get("id") or None #meshID #unless this is supposed to be unique?
                            # yes should have a none-MESH id 
                            ann_type = stype or None
                            loc_offset = None
                            loc_len = None
                            #print("Found annotation: " + ann)
                            
                            # if ann_text: 
                            #     if termA in ann_text.lower():
                            #         print("Search Term A found in annotation in sentence, in")
                            #     if termC in ann_text.lower():
                            #         print("Search Term C found in annotation in sentence, in")

                            loc_offset = get_child_text(sentence, "offset") or None
                            loc_len = get_child_text(sentence, "length") or None
                            # for loc in get_children(ann, "location"):
                            #     if "offset" in loc.attrib:
                            #         try:
                            #             loc_offset = int(loc.attrib.get("offset"))
                            #         except Exception:
                            #             loc_offset = None
                            #     if "length" in loc.attrib:
                            #         try:
                            #             loc_len = int(loc.attrib.get("length"))
                            #         except Exception:
                            #             loc_len = None
                            #     if loc_offset is None and loc.text and loc.text.isdigit():
                            #         loc_offset = int(loc.text)
                            cur.execute(
                                "INSERT OR REPLACE INTO annotations(ann_id, doc_id, passage_id, sentence_id, location_offset, location_length, type, text, infons) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)",
                                (ann_id, doc_id, passage_rowid, sentence_rowid, loc_offset, loc_len, ann_type, ann_text, json.dumps(ann_infons, ensure_ascii=False))
                            )
                            
                        # relations inside sentence
                        for rel in (r for r in sentence if stype.lower() in ("relation")):
                            rel_infons = s_infons
                            rel_id = rel_infons.get("identifier") or rel_infons.get("id") or rel_infons.get("rel_id") or get_child_text(rel, "id") or None
                            #rel_infons = infons_to_dict(rel)
                            print("Found relation: " + rel)
                            # shouldddd have access to passage_rowid and sentence_rowid - why not? 
                            cur.execute(
                                "INSERT OR REPLACE INTO relations(rel_id, doc_id, passage_id, sentence_id, infons) VALUES (?, ?, ?, ?, ?)",
                                (rel_id, doc_id, passage_rowid, sentence_rowid, json.dumps(rel_infons, ensure_ascii=False))
                            )
                            for node in get_children(rel, "node"):
                                refid = node.get("refid")
                                role = node.get("role")
                                cur.execute(
                                    "INSERT INTO relation_nodes(rel_id, refid, role) VALUES (?, ?, ?)",
                                    (rel_id, refid, role)
                                )

                    # annotations directly under passage
                    for ann in (a for a in passage if ptype.lower() in ("chemical species disease gene annotation")):
                        ann_text = get_child_text(ann, "text") or None
                        ann_infons = infons_to_dict(ann) 
                        ann_id = get_child_text(ann, "id") or get_child_text(ann, "identifier") or None
                        #ann_id = ann_infons.get("identifier") or ann_infons.get("id") or None #meshID #unless this is supposed to be unique?
                        ann_type = ptype or None
                        loc_offset = None
                        loc_len = None

                        if termA in ann_text.lower():
                            print("Search Term A found in annotation under passage")
                        if termC in ann_text.lower():
                            print("Search Term C found in annotation under passage")
                        
                        for loc in get_children(ann, "location"):
                            if "offset" in loc.attrib:
                                try:
                                    loc_offset = int(loc.attrib.get("offset"))
                                except Exception:
                                    loc_offset = None
                            if "length" in loc.attrib:
                                try:
                                    loc_len = int(loc.attrib.get("length"))
                                except Exception:
                                    loc_len = None
                            if loc_offset is None and loc.text and loc.text.isdigit():
                                loc_offset = int(loc.text)
                        cur.execute(
                                "INSERT OR REPLACE INTO annotations(ann_id, doc_id, passage_id, sentence_id, location_offset, location_length, type, text, infons) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)",
                                (ann_id, doc_id, passage_rowid, sentence_rowid, loc_offset, loc_len, ann_type, ann_text, json.dumps(ann_infons, ensure_ascii=False))
                        )

                    # relations directly under passage
                    for rel in (r for r in passage if ptype.lower() in "relation"):
                        rel_id = get_child_text(rel, "id") or None
                        rel_infons = infons_to_dict(rel)
                        cur.execute(
                            "INSERT OR REPLACE INTO relations(rel_id, doc_id, passage_id, sentence_id, infons) VALUES (?, ?, ?, ?, ?)",
                            (rel_id, doc_id, passage_rowid, None, json.dumps(rel_infons, ensure_ascii=False))
                        )
                        for node in get_children(rel, "node"):
                            refid = node.get("refid")
                            role = node.get("role")
                            cur.execute(
                                "INSERT INTO relation_nodes(rel_id, refid, role) VALUES (?, ?, ?)",
                                (rel_id, refid, role)
                            )

                # annotations/relations directly under document
                for ann in (a for a in elem if localname(a.tag) == "annotation"):
                    print("annotation found under document")
                    ann_text = get_child_text(ann, "text") or None
                    ann_infons = infons_to_dict(ann) 
                    ann_id = get_child_text(ann, "id") or get_child_text(ann, "identifier") or None
                    #ann_id = ann_infons.get("identifier") or ann_infons.get("id") or None #meshID #unless this is supposed to be unique?
                    ann_type = doc_type or None
                    loc_offset = None
                    loc_len = None

                    if termA in ann_text.lower():
                        print("Search Term A found in annotation under document")
                    if termC in ann_text.lower():
                        print("Search Term C found in annotation under document")
                    
                    for loc in get_children(ann, "location"):
                        if "offset" in loc.attrib:
                            try:
                                loc_offset = int(loc.attrib.get("offset"))
                            except Exception:
                                loc_offset = None
                        if "length" in loc.attrib:
                            try:
                                loc_len = int(loc.attrib.get("length"))
                            except Exception:
                                loc_len = None
                    cur.execute(
                                "INSERT OR REPLACE INTO annotations(ann_id, doc_id, passage_id, sentence_id, location_offset, location_length, type, text, infons) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
                                (ann_id, doc_id, passage_rowid, sentence_rowid, loc_offset, loc_len, ann_type, ann_text, json.dumps(ann_infons, ensure_ascii=False))
                    )

                for rel in (r for r in elem if localname(r.tag) == "relation"):
                    #print("relation found under document")
                    # is working!! need member limit more than 2 
                    # not getting rel_id though
                    rel_id = get_child_text(rel, "id") or None
                    if "id" in rel.tag:
                        print("rel.text: " + rel.text) #not printing this
                    rel_infons = infons_to_dict(rel)
                    cur.execute(
                        "INSERT OR REPLACE INTO relations(rel_id, doc_id, passage_id, sentence_id, infons) VALUES (?, ?, ?, ?, ?)",
                        (rel_id, doc_id, None, None, json.dumps(rel_infons, ensure_ascii=False))
                    )
                    for node in get_children(rel, "node"):
                        refid = node.get("refid")
                        role = node.get("role")
                        cur.execute(
                            "INSERT INTO relation_nodes(rel_id, refid, role) VALUES (?, ?, ?)",
                            (rel_id, refid, role)
                        )
                            
                if title_text or abstract_text:
                    cur.execute("UPDATE documents SET title = ?, abstract = ?, contains_a = ?, contains_c = ? WHERE doc_id = ?", (title_text, abstract_text, contains_a, contains_c, doc_id))

                if contains_a > 0:
                    if contains_c > 0:
                        cur.execute("INSERT OR REPLACE INTO overlap_documents(doc_id, member_name, infons, title, abstract, doc_type, author, contains_a, contains_c) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)",
                    (doc_id, member_name, json.dumps(doc_infons, ensure_ascii=False), title_text, abstract_text, doc_type, author, contains_a, contains_c)
                    ) 
                    else:
                        cur.execute("INSERT OR REPLACE INTO a_documents(doc_id, member_name, infons, title, abstract, doc_type, author, contains_a, contains_c) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)",
                    (doc_id, member_name, json.dumps(doc_infons, ensure_ascii=False), title_text, abstract_text, doc_type, author, contains_a, contains_c)
                    )
                        
                elif contains_c > 0:
                    cur.execute("INSERT OR REPLACE INTO c_documents(doc_id, member_name, infons, title, abstract, doc_type, author, contains_a, contains_c) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)",
                    (doc_id, member_name, json.dumps(doc_infons, ensure_ascii=False), title_text, abstract_text, doc_type, author, contains_a, contains_c)
                    )
                    
                docs += 1
                if docs % docs_commit_batch == 0:
                    conn.commit()
                    if show_progress:
                        elapsed = time.time() - t0
                        print(f"  inserted {docs} docs from member {member_name} (elapsed {elapsed:.1f}s)")

                # free memory
                elem.clear()
                while elem.getprevious() is not None:
                    del elem.getparent()[0]

                if max_docs is not None and docs >= max_docs:
                    break
    finally:
        # final commit for this member
        conn.commit()
        try:
            context.close()
        except Exception:
            pass

    return docs

### results from passage p_infons
Processing member: output/BioCXML/10.BioC.XML
{'name_3': 'surname:Richert;given-names:Agnieszka', 'name_2': 'surname:Degli Esposti;given-names:Micaela', 'name_1': 'surname:Ferri;given-names:Martina', 'name_0': 'surname:Olewnik-Kruszkowska;given-names:Ewa', 'issue': '15', 'year': '2025', 'article-id_pmc': '12349510', 'article-id_publisher-id': 'polymers-17-02072', 'article-id_doi': '10.3390/polym17152072', 'type': 'front', 'elocation-id': '2072', 'volume': '17', 'license': 'Licensee MDPI, Basel, Switzerland. This article is an open access article distributed under the terms and conditions of the Creative Commons Attribution (CC BY) license (https://creativecommons.org/licenses/by/4.0/).', 'section_type': 'TITLE', 'kwd': 'cellulose derivatives surfactants phenolic acids protection of food active packaging', 'name_7': 'surname:Vodnar;given-names:Dan Cristian', 'name_6': 'surname:Tosa;given-names:Monica Ioana', 'name_5': 'surname:Martau;given-names:Gheorghe', 'name_4': 'surname:Fabbri;given-names:Paola'}
{'section_type': 'ABSTRACT', 'type': 'abstract'}
{'section_type': 'INTRO', 'type': 'title_1'}
{'section_type': 'INTRO', 'type': 'paragraph'}
{'section_type': 'INTRO', 'type': 'paragraph'}
{'section_type': 'INTRO', 'type': 'paragraph'}
{'section_type': 'INTRO', 'type': 'paragraph'}
{'section_type': 'INTRO', 'type': 'paragraph'}
{'section_type': 'INTRO', 'type': 'paragraph'}
{'section_type': 'METHODS', 'type': 'title_1'}
{'section_type': 'METHODS', 'type': 'title_2'}
{'section_type': 'METHODS', 'type': 'paragraph'}

### results from paragraph / sentence s_infons
Processing member: output/BioCXML/10.BioC.XML
{}
{}
{}
{}
{'identifier': 'MESH:D010636', 'type': 'Chemical'}
{'identifier': 'MESH:D013729', 'type': 'Chemical'}
{'identifier': 'MESH:D005419', 'type': 'Chemical'}
{'identifier': '-', 'type': 'Chemical'}
{'identifier': 'MESH:D009822', 'type': 'Chemical'}
{}
{}
{}
{}
{'identifier': 'MESH:D048271', 'type': 'Chemical'}
{'identifier': '-', 'type': 'Chemical'}
{'identifier': 'MESH:D009822', 'type': 'Chemical'}
{'identifier': 'MESH:D048271', 'type': 'Chemical'}
{'identifier': '-', 'type': 'Chemical'}
{'identifier': 'MESH:D002482', 'type': 'Chemical'}
{'identifier': 'MESH:C033616', 'type': 'Chemical'}
{'identifier': 'MESH:D011794', 'type': 'Chemical'}
{'identifier': 'MESH:C073473', 'type': 'Chemical'}
{'identifier': 'MESH:C073473', 'type': 'Chemical'}
{'identifier': 'MESH:D011136', 'type': 'Chemical'}
{'identifier': 'MESH:C029010', 'type': 'Chemical'}

### results from sentence sqlite
sentences
                id  passage_id  \
0                1           4   
1                2           4   
2                3           4   
3                4           4   
4                5           4   
...            ...         ...   
18519412  18519413     8251490   
18519413  18519414     8251490   
18519414  18519415     8251490   
18519415  18519416     8251490   
18519416  18519417     8251490   

                                                     infons        text  
0                                                        {}              
1                                                        {}              
2                                                        {}              
3                                                        {}              
4         {"identifier": "MESH:D010636", "type": "Chemic...     phenols  
...                                                     ...         ...  
18519412            {"identifier": "-", "type": "Chemical"}     CH2CHCH  
18519413  {"identifier": "MESH:D002244", "type": "Chemic...           C  
18519414  {"identifier": "MESH:D010672", "type": "Chemic...  C15H12N2O2  
18519415            {"identifier": "-", "type": "Chemical"}          1H  
18519416  {"identifier": "MESH:D009584", "type": "Chemic...           N 

## Stream Tar and Process

In [8]:
# Stream the tar archive, process each member, and record progress for resume
def stream_tar_and_process(url, termA, termC: str, conn: sqlite3.Connection,
                           member_limit: Optional[int] = MEMBER_LIMIT,
                           docs_commit_batch: int = DOCS_COMMIT_BATCH,
                           process_only_xml: bool = PROCESS_ONLY_XML,
                           resume: bool = True):
    """
    Streams the tar.gz at url and processes members sequentially.
    If resume=True, members listed in processed_members table are skipped.
    """
    # Get set of already processed members
    cur = conn.cursor()
    if resume:
        cur.execute("SELECT member_name FROM processed_members")
        processed_set = set(r[0] for r in cur.fetchall())
    else:
        processed_set = set()

    resp = requests.get(url, stream=True, timeout=60)
    resp.raise_for_status()
    resp.raw.decode_content = True
    # choose streaming mode; let tarfile detect compression
    tar = tarfile.open(fileobj=resp.raw, mode="r|*")

    members_done = 0
    total_docs = 0
    t0_all = time.time()

    try:
        for member in tar:
            if member_limit is not None and members_done >= member_limit:
                break
            if not member.isfile():
                continue
            name = member.name
            if resume and name in processed_set:
                if SHOW_PROGRESS:
                    print(f"Skipping already-processed member: {name}")
                members_done += 1
                continue
            # Optionally skip non-XML files
            if process_only_xml and not (name.lower().endswith(".xml") or name.lower().endswith(".xml.gz") or name.lower().endswith(".bioc") or name.lower().endswith(".bioc.gz")):
                if SHOW_PROGRESS:
                    print(f"Skipping non-XML member: {name}")
                members_done += 1
                # we don't mark non-XML members as processed to allow future runs to reconsider them
                continue

            if SHOW_PROGRESS:
                print(f"Processing member: {name}")

            fobj = tar.extractfile(member)
            if fobj is None:
                print(f"  [WARN] could not extract {name}")
                members_done += 1
                continue

            try:
                docs = parse_member_and_insert(fobj, name, termA, termC, conn, docs_commit_batch)
            except Exception as e:
                # On errors, commit what we have and raise or continue based on policy.
                conn.commit()
                print(f"  [ERROR] parsing member {name}: {e}", file=sys.stderr)
                # Option: mark as failed by not recording processed_members so you can retry later.
                # We'll re-raise to stop unless you prefer to continue
                raise
            finally:
                try:
                    fobj.close()
                except Exception:
                    pass

            # record that member completed
            cur.execute("INSERT OR REPLACE INTO processed_members(member_name, processed_at) VALUES (?, ?)", (name, time.time()))
            conn.commit()

            members_done += 1
            total_docs += docs
            if SHOW_PROGRESS:
                print(f"  finished member {name}: {docs} documents")

    finally:
        try:
            tar.close()
        except Exception:
            pass
        try:
            resp.close()
        except Exception:
            pass

    elapsed = time.time() - t0_all
    print(f"Completed {members_done} members, {total_docs} documents in {elapsed:.1f}s")

## Make sqlite file into pandas dataframe then graph

In [ ]:
def make_dataframe (termA: str, termC: str, conn: sqlite3.Connection):
    ## ALL DOCUMENTS READ IN ##
    docs = pd.read_sql('SELECT doc_id, title, abstract, contains_a, contains_c FROM documents', conn)
    print("********************")
    print("documents")
    print(docs)
    print(pd.read_sql('SELECT SUM(contains_a) FROM documents', conn))
    print(pd.read_sql('SELECT SUM(contains_c) FROM documents', conn))

    ## DOCUMENTS WITH SEARCH TERM A ##
    docs_a = pd.read_sql('SELECT doc_id, title FROM a_documents', conn)
    print("********************")
    print("documents containing only search term A: " + termA)
    print(docs_a)

    ## DOCUMENTS WITH SEARCH TERM C ##
    docs_c = pd.read_sql('SELECT doc_id, title FROM c_documents', conn)
    print("********************")
    print("documents containing only search term C: " + termC)
    print(docs_c)

    ## DOCUMENTS WITH BOTH SEARCH TERMS ## (notable, but to be ignored)
    print("********************")
    print("documents containing both search terms (overlapping, not considered for B)")
    print(pd.read_sql('SELECT doc_id, title FROM overlap_documents', conn))
    #print(pd.read_sql('SELECT COUNT(DISTINCT title) FROM documents', conn))
    #print("sqlite_master")
    #print(pd.read_sql('SELECT * FROM sqlite_master', conn))

    ## SENTENCES / TEXT / PASSAGES ##
    print("********************")
    print("sentences")
    print(pd.read_sql('SELECT * FROM sentences', conn)) 
    #print(pd.read_sql('SELECT COUNT(DISTINCT passage_id) FROM sentences', conn))
    
    ## ANNOTATIONS ##
    print("annotations")
    print(pd.read_sql('SELECT * FROM annotations', conn))
    #print(pd.read_sql('SELECT COUNT(DISTINCT ann_id) FROM annotations', conn))
    #print(pd.read_sql('SELECT COUNT(DISTINCT doc_id) FROM annotations', conn))
    #print(pd.read_sql('SELECT COUNT(doc_id) FROM annotations', conn))
    
    ## RELATIONS ##
    print("relations")
    relations = pd.read_sql('SELECT * FROM relations', conn)
    print(relations)
    print(pd.read_sql('SELECT COUNT(DISTINCT rel_id) FROM relations', conn))
    print("relation_nodes")
    relation_nodes = pd.read_sql('SELECT * FROM relation_nodes', conn)
    print(relation_nodes)
        #role is a pair of numbers eg 2,5
    make_graph(relations, relation_nodes)

In [10]:
def make_graph(relations, relation_nodes):
    #print(relations)
    #print(relation_nodes)
    #will need to iterate through these
    #pairs = tuple(relation_nodes['role'][1].split(','))
    #then iterate through and convert all to numbers?? can't just int() a tuple or list 
    #print(pairs)
    G = nx.Graph()
    G.add_nodes_from(relation_nodes)
    G.add_nodes_from(relations['doc_id'])
    G.add_nodes_from(relation_nodes['refid'])
    #rel_tuples = list(relations.itertuples(index=False, name=None))
    # nodes are rel_id; labels are role; not sure what refid is 
    # G.add_edges_from(relation_nodes) needs to be 2 tuple or 3 tuple - try splitting at , into tuple?
    #G.add_edges_from(rel_tuples) #source is doc_id, target is rel_id. needs to be tuple
    #G.add_edges_from(pairs)
    nx.draw(G, with_labels=True)
    plt.title("Connecting relations")
    plt.show()

### note on next steps:
for connections / identifying 0-hops getting and comparing authors would be really helpful 
are they saved in the file??
get from passages -> name_3 -> surname, given-names

## Main

In [ ]:
# Main: initialize DB and run the stream processing
if __name__ == "__main__":
    # user input / instructions
    searchTermA = input("Input search term A, default is Migraine: ").lower() or "Migraine"
    searchTermC = input("Input search term C, default is Magnesium: ").lower() or "Magnesium"
    # if no inputs or not correct format, set to Migraine and Magnesium as defaults 
    print("Searching for the following terms: " + searchTermA + " and " + searchTermC)
    # need some way to check if had already been working with / downloading specifically those terms - save in Element or document SQL? 
    
    # Create DB if needed
    conn = init_db(SQLITE_PATH)

    # Timer should start here (use tqdm) 

    # Access BioCXML Files from pubtator3 
    try:
        # Iterate through all 
        i = 0
        while i < len(URLS):
            stream_tar_and_process(URLS[i], searchTermA, searchTermC, conn, member_limit=MEMBER_LIMIT)
            i+=1
        # Or use just the first one for faster testing
        #stream_tar_and_process(URL, searchTermA, searchTermC, conn, member_limit=MEMBER_LIMIT)

        # Make results into a Pandas dataframe, and then a network graph, and then a visualization of that network graph 
        make_dataframe(searchTermA, searchTermC, conn)
    finally:
        conn.close()

# will need to delete files and re-run all once processing pipeline is complete - early members missing steps
# restart kernal after deleting files, else "disc I/O" errors

Input search term A, default is Migraine:  Migraine
Input search term C, default is Magnesium:  Magnesium


Searching for the following terms: migraine and magnesium
Skipping already-processed member: output/BioCXML/10.BioC.XML
Skipping already-processed member: output/BioCXML/100.BioC.XML
Skipping already-processed member: output/BioCXML/1000.BioC.XML
Skipping already-processed member: output/BioCXML/10000.BioC.XML
Skipping already-processed member: output/BioCXML/100000.BioC.XML
Skipping already-processed member: output/BioCXML/100010.BioC.XML
Skipping already-processed member: output/BioCXML/100020.BioC.XML
Skipping already-processed member: output/BioCXML/100030.BioC.XML
Skipping already-processed member: output/BioCXML/100040.BioC.XML
Skipping already-processed member: output/BioCXML/100050.BioC.XML
Completed 10 members, 0 documents in 0.2s
Processing member: output/BioCXML/1.BioC.XML
  finished member output/BioCXML/1.BioC.XML: 100 documents
Processing member: output/BioCXML/100001.BioC.XML
  inserted 200 docs from member output/BioCXML/100001.BioC.XML (elapsed 0.1s)
  inserted 400 doc

In [16]:
# %% [markdown]
# Quick queries and tips
# - To inspect progress:
#     sqlite3 pubtator_bioc0.sqlite "SELECT COUNT(*) FROM processed_members;"
# - To see total documents imported:
#     sqlite3 pubtator_bioc0.sqlite "SELECT COUNT(*) FROM documents;"
# - If you want a different schema (e.g., normalized annotation types in columns), tell me which infon keys you care about and I can adapt the script to extract them into dedicated columns.
# - If you'd like the script to be more robust to intermittent network failures (auto-retry, resume after transient errors), I can add retry logic and exponential backoff.
#
# Run this notebook locally. Start with MEMBER_LIMIT small (e.g., 1 or 5) to confirm behavior, then set to None to process the entire archive.